# Test Set Evaluation

Compare classical baselines (MLP, LSTM) and the quantum reservoir computing (QRC) ensemble
on held-out test data that was never seen during training or model selection.

## What's inside

| Section | Description |
|:--------|:------------|
| §1 Setup | Load training data, fit scaler, load test data (6 days × 224 instruments) |
| §2 Load Models | Re-define MLP/LSTM architectures, load saved weights, generate test predictions |
| §3 Metrics Table | Validation vs test metrics (RMSE, MAE, QLIKE) for all 3 models |
| §4 QRC Evaluation | Load QRC ensemble predictions, add to comparison table |
| §5 Per-Day QLIKE | Day-by-day QLIKE breakdown across all models |
| §6 Plots | Visual comparison of predictions vs actuals on 6 sample instruments |

## Test setup

- **Test data:** 6 days (24 Dec 2051 – 01 Jan 2052), 224 swaption instruments
- **Input:** Last 20 training days → models predict 10 days → evaluate first 6
- **QRC predictions:** Ensemble of top-3 reservoirs by validation QLIKE (see QRC notebook)

## Final results

| Model | Test QLIKE | Test RMSE |
|:------|:----------:|----------:|
| MLP | 0.003921 | 0.015962 |
| LSTM | 0.001081 | 0.007712 |
| **QRC ensemble** | **0.000771** | **0.006693** |

> **QRC ensemble beats LSTM by 29% on test QLIKE.**

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
# --- Config (must match training notebook) ---
WINDOW = 20
HORIZON = 10
VAL_SIZE = 30
DEVICE = "cpu"

# --- Load training data (need scaler + last window as model input) ---
train_df = pd.read_parquet("../data/level1.parquet")
train_df['Date'] = pd.to_datetime(train_df['Date'], format='mixed')
train_df = train_df.sort_values('Date').reset_index(drop=True)
price_cols = [c for c in train_df.columns if c != 'Date']
prices = train_df[price_cols].astype(float).values  # (494, 224)

# TODO: refactor — scaler fitting is duplicated from Classical_prediction_baseline.ipynb
# Could save/load scaler with joblib, or move to a shared utils.py
scaler = StandardScaler()
scaler.fit(prices[:-VAL_SIZE])
prices_scaled = scaler.transform(prices)

# --- Load test data (reorder cols to match training) ---
test_df = pd.read_excel("../data/test.xlsx")
test_df['Date'] = pd.to_datetime(test_df['Date'], format='mixed')
test_prices = test_df[price_cols].astype(float).values  # (6, 224)
N_TEST = len(test_prices)

print(f"Train: {prices.shape}, Test: {test_prices.shape}")
print(f"Test dates: {list(test_df['Date'].dt.strftime('%Y-%m-%d'))}")

## Load Models and Generate Predictions

In [ ]:
# TODO: refactor — model classes are duplicated from Classical_prediction_baseline.ipynb
# Move MLP, LSTMBaseline to a shared models.py and import in both notebooks.
# We MUST define them here because torch.load(state_dict) requires the class to exist.

in_dim, out_dim = WINDOW * 224, HORIZON * 224

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x): return self.net(x)

class LSTMBaseline(nn.Module):
    def __init__(self, input_dim=224, hidden_dim=64, num_layers=1, output_dim=2240):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1])

# Load saved weights
mlp_model = MLP(in_dim, out_dim).to(DEVICE)
mlp_model.load_state_dict(torch.load("../models/mlp_best.pt", weights_only=True))
mlp_model.eval()

lstm_model = LSTMBaseline().to(DEVICE)
lstm_model.load_state_dict(torch.load("../models/lstm_best.pt", weights_only=True))
lstm_model.eval()

# --- Predict on test (input = last WINDOW days of training data) ---
last_window = prices_scaled[-WINDOW:]

with torch.no_grad():
    mlp_input = torch.tensor(last_window.flatten(), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    mlp_pred = scaler.inverse_transform(
        mlp_model(mlp_input).cpu().numpy().reshape(HORIZON, 224)
    )[:N_TEST]

    lstm_input = torch.tensor(last_window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    lstm_pred = scaler.inverse_transform(
        lstm_model(lstm_input).cpu().numpy().reshape(HORIZON, 224)
    )[:N_TEST]

actual = test_prices
print(f"Predictions: MLP {mlp_pred.shape}, LSTM {lstm_pred.shape}, Actual {actual.shape}")

## Validation vs Test Metrics

In [ ]:
# TODO: refactor — compute_metrics duplicated from baseline nb, move to utils.py
def compute_metrics(pred, actual):
    rmse = np.sqrt(((pred - actual) ** 2).mean())
    mae = np.abs(pred - actual).mean()
    eps = 1e-8
    ratio = actual / np.clip(pred, eps, None)
    qlike = (ratio - np.log(ratio) - 1).mean()
    return {'RMSE': rmse, 'MAE': mae, 'QLIKE': qlike}

# Test metrics
mlp_test = compute_metrics(mlp_pred, actual)
lstm_test = compute_metrics(lstm_pred, actual)

# Val metrics from training notebook (hardcoded — already computed there)
mlp_val = {'RMSE': 0.0162, 'MAE': 0.0120, 'QLIKE': 0.003752}
lstm_val = {'RMSE': 0.0155, 'MAE': 0.0116, 'QLIKE': 0.003293}

# Build comparison table
rows = []
for metric in ['RMSE', 'MAE', 'QLIKE']:
    rows.append({
        'Metric': metric,
        'MLP (val)': mlp_val[metric],
        'MLP (test)': mlp_test[metric],
        'LSTM (val)': lstm_val[metric],
        'LSTM (test)': lstm_test[metric],
    })

results = pd.DataFrame(rows)
results.style.format({c: '{:.6f}' for c in results.columns if c != 'Metric'}).set_caption(
    'Classical Baselines: Validation vs Test'
)

# QRC Evaluation

In [ ]:
# --- QRC Test Evaluation ---
qrc_pred = np.load("../models/qrc_test_pred.npy")  # (6, 224)
qrc_test = compute_metrics(qrc_pred, actual)
qrc_val = {'RMSE': 0.017487, 'MAE': 0.013090, 'QLIKE': 0.004282}  # ensemble top-3 by val QLIKE

# Add to results table
results['QRC (val)'] = [qrc_val[m] for m in ['RMSE', 'MAE', 'QLIKE']]
results['QRC (test)'] = [qrc_test[m] for m in ['RMSE', 'MAE', 'QLIKE']]
results.style.format({c: '{:.6f}' for c in results.columns if c != 'Metric'}).set_caption(

    'All Models: Validation vs Test')

In [ ]:
# --- Per-day QLIKE for each model ---
eps = 1e-8
test_dates = list(test_df['Date'].dt.strftime('%Y-%m-%d'))

def qlike_per_day(pred, actual):
    """QLIKE averaged across 224 instruments, for each test day."""
    ratio = actual / np.clip(pred, eps, None)
    return (ratio - np.log(ratio) - 1).mean(axis=1)  # (N_TEST,)

qlike_rows = []
for i, date in enumerate(test_dates):
    qlike_rows.append({
        'Date': date,
        'MLP': qlike_per_day(mlp_pred, actual)[i],
        'LSTM': qlike_per_day(lstm_pred, actual)[i],
        'QRC': qlike_per_day(qrc_pred, actual)[i],
    })

qlike_df = pd.DataFrame(qlike_rows)
# Add mean row
qlike_df.loc[len(qlike_df)] = ['Mean'] + [qlike_df[c].mean() for c in ['MLP', 'LSTM', 'QRC']]

qlike_df.style.format({c: '{:.6f}' for c in ['MLP', 'LSTM', 'QRC']}).set_caption(
    'QLIKE per Test Day (lower is better)'
).apply(lambda row: ['font-weight: bold' if row['Date'] == 'Mean' else '' for _ in row], axis=1)

## Test Predictions vs Actuals

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sample_cols = [0, 50, 100, 150, 200, 223]

for ax, col_idx in zip(axes.flat, sample_cols):
    col_name = price_cols[col_idx]
    ctx = prices[-20:, col_idx]  # last 20 training days as context
    act = actual[:, col_idx]
    x_ctx, x_test = range(20), range(20, 20 + N_TEST)
    
    ax.plot(x_ctx, ctx, 'b-', alpha=0.5, label='Train')
    ax.plot(x_test, act, 'b-o', markersize=3, label='Actual')
    ax.plot(x_test, mlp_pred[:, col_idx], 'r--s', markersize=3, label='MLP')
    ax.plot(x_test, lstm_pred[:, col_idx], 'g--^', markersize=3, label='LSTM')
    ax.plot(x_test, qrc_pred[:, col_idx], 'm--D', markersize=3, label='QRC')
    ax.axvline(19.5, color='gray', ls=':', alpha=0.5)
    ax.set_title(col_name, fontsize=9)
    ax.legend(fontsize=6)
plt.suptitle('Test Set: Predictions vs Actual', fontsize=13)
plt.tight_layout()
plt.show()